In [1]:
import os
import re
import pdfplumber
import pandas as pd
from pathlib import Path
from dateutil import parser as dateparser

In [2]:
KNOWN_BRANDS = [
    'petron', 'shell', 'caltex', 'phoenix', 'total',
    'flying v', 'flyingv', 'unioil', 'seaoil', 'ptt', 'independent'
]

PRODUCTS = [
    'ron 100', 'ron 97', 'ron 95', 'ron 91',
    'diesel plus', 'diesel', 'kerosene',
    'gasoline (ron 97)', 'gasoline (ron 95)',
    'gasoline (ron 91)', 'gasoline (ron 100)',
    'premium plus', 'premium'
]

SKIP_COLS = ['overall', 'range', 'common', 'price', 'area', 'product', 'cities']

In [3]:
def parse_range(value):
    """Split 'XX.XX-XX.XX' string into (min, max) floats."""
    if not value or str(value).strip() in ('', '#N/A', 'NONE', 'N/A', '-'):
        return None, None
    value = str(value).strip()
    match = re.match(r'([\d.]+)\s*[-–]\s*([\d.]+)', value)
    if match:
        return float(match.group(1)), float(match.group(2))
    try:
        f = float(value)
        return f, f
    except ValueError:
        return None, None

In [5]:
def clean_cell(val):
    """Normalize a cell value."""
    if val is None:
        return None
    cleaned = str(val).strip().replace('\n', ' ')
    return cleaned if cleaned else None


In [6]:
def is_product_row(val):
    """Check if a string matches a known fuel product."""
    if not val:
        return False
    return any(p in str(val).strip().lower() for p in PRODUCTS)


In [7]:
def is_brand(val):
    """Check if a string matches a known brand."""
    if not val:
        return False
    return any(b in str(val).strip().lower() for b in KNOWN_BRANDS)

In [8]:

def should_skip_col(val):
    """Check if a column header should be skipped when mapping brands."""
    if not val:
        return False
    return any(s in str(val).strip().lower() for s in SKIP_COLS)


In [9]:
def classify_table(table):
    """
    Returns:
        'detail'   - main area x product x brand table (Type D/E)
        'summary'  - NCR-wide prevailing prices summary block
        'type_a'   - 2017 split brand table with range strings
        'common'   - 2017 standalone common price block
        'single'   - single non-area table (Type B/C)
        'unknown'
    """
    if not table or len(table) < 2:
        return 'unknown'

    flat = ' '.join(
        str(c).lower()
        for row in table[:3]
        for c in row if c
    )

    if 'area' in flat and 'product' in flat:
        return 'detail'
    if 'prevailing retail' in flat and 'area' not in flat and len(table) < 10:
        return 'summary'
    if 'common price' in flat and 'ron' in flat and 'area' not in flat and len(table) < 8:
        return 'common'
    if 'product' in flat and any(b in flat for b in KNOWN_BRANDS):
        return 'type_a' if re.search(r'\d+\.\d+-\d+\.\d+', flat) else 'single'
    return 'unknown'

In [10]:
def extract_effectivity_date(pdf_path):
    """
    Extracts the effectivity date from a DOE fuel price bulletin.
    Handles all 5 format types.

    Patterns by type:
      A/B/C : "Effectivity: February 07, 2017"
      D     : "Date of Effectivity: June 11, 2019"
      E     : "For the week of January 6-12, 2026"  → takes start date
              "For the period of March 5-11, 2024"  → takes start date
              "For the period of May 28 - Jun 3, 2024" → takes start date
    """
    with pdfplumber.open(pdf_path) as pdf:
        # Scan all pages for the date (usually page 1, sometimes page 2)
        full_text = ''
        for page in pdf.pages:
            full_text += (page.extract_text() or '') + '\n'

    # --- Pattern 1: "Effectivity: <date>" or "Date of Effectivity: <date>" ---
    match = re.search(
        r'(?:Date of\s+)?Effectivity\s*[:\-]\s*([A-Za-z]+\s+\d{1,2},?\s*\d{4})',
        full_text, re.IGNORECASE
    )
    if match:
        try:
            return dateparser.parse(match.group(1)).date()
        except Exception:
            pass

    # --- Pattern 2: "For the week/period of <date range>" → take start date ---
    match = re.search(
        r'For the (?:week|period) of\s+'
        r'([A-Za-z]+\s+\d{1,2})\s*[-–]\s*(?:[A-Za-z]+\s+)?\d{1,2},?\s*(\d{4})',
        full_text, re.IGNORECASE
    )
    if match:
        try:
            # Combine start day + year e.g. "January 6" + "2026"
            raw = f"{match.group(1)} {match.group(2)}"
            return dateparser.parse(raw).date()
        except Exception:
            pass

    # --- Pattern 3: "As of <date>" fallback ---
    match = re.search(
        r'As of\s*[:\-]?\s*([A-Za-z]+\s+\d{1,2},?\s*\d{4})',
        full_text, re.IGNORECASE
    )
    if match:
        try:
            return dateparser.parse(match.group(1)).date()
        except Exception:
            pass

    # --- Pattern 4: "NCR Prevailing Retail Pump Price (As of: <date>)" ---
    match = re.search(
        r'As of\s*:\s*([A-Za-z]+\s+\d{1,2},?\s*\d{4})',
        full_text, re.IGNORECASE
    )
    if match:
        try:
            return dateparser.parse(match.group(1)).date()
        except Exception:
            pass

    print(f"  ⚠ Could not extract effectivity date from {Path(pdf_path).name}")
    return None

### Parsing

In [11]:
def parse_type_a(pdf_path):
    """
    Type A - 2017 early:
    Multiple small brand-group tables per page, products as rows,
    values are 'XX.XX-XX.XX' range strings. No area breakdown.
    Optional separate common price block below.
    """
    all_brand_dfs = []
    df_summary = None

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            tables = page.extract_tables({
                "vertical_strategy": "text",
                "horizontal_strategy": "text",
                "snap_tolerance": 5,
            })

            for table in tables:
                kind = classify_table(table)

                # --- Standalone common price block ---
                if kind == 'common':
                    rows = []
                    for row in table[1:]:
                        cells = [clean_cell(c) for c in row if clean_cell(c)]
                        if len(cells) >= 2:
                            try:
                                rows.append({
                                    'product': cells[0],
                                    'common_price': float(cells[1])
                                })
                            except ValueError:
                                pass
                    if rows:
                        df_summary = pd.DataFrame(rows)
                    continue

                if kind not in ('type_a', 'single'):
                    continue

                # --- Brand price table ---
                headers = [clean_cell(c) for c in table[0]]
                product_col = next(
                    (i for i, h in enumerate(headers)
                     if h and 'product' in h.lower()), 0
                )
                brand_cols = [
                    (i, headers[i]) for i in range(len(headers))
                    if i != product_col and headers[i] and is_brand(headers[i])
                ]

                records = []
                for row in table[1:]:
                    product = clean_cell(row[product_col]) if product_col < len(row) else None
                    if not is_product_row(product):
                        continue
                    for col_idx, brand in brand_cols:
                        val = clean_cell(row[col_idx]) if col_idx < len(row) else None
                        price_min, price_max = parse_range(val)
                        if price_min or price_max:
                            records.append({
                                'product': product.upper(),
                                'brand': brand.upper(),
                                'price_min': price_min,
                                'price_max': price_max,
                            })

                if records:
                    all_brand_dfs.append(pd.DataFrame(records))

    if not all_brand_dfs:
        return None, df_summary

    df_detail = pd.concat(all_brand_dfs, ignore_index=True)
    df_detail = df_detail.dropna(subset=['price_min', 'price_max'], how='all')
    return df_detail, df_summary

In [12]:
def parse_type_b(pdf_path):
    """
    Type B - 2017 mid:
    Single table, no area breakdown, paired brand columns (min, max).
    NO Overall Range or Common Price inside the table.
    Common prices live in a separate block below the table.
    """
    records = []
    df_summary = None

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            tables = page.extract_tables({
                "vertical_strategy": "text",
                "horizontal_strategy": "text",
                "snap_tolerance": 5,
            })

            for table in tables:
                if not table or len(table) < 2:
                    continue

                header = [clean_cell(c) for c in table[0]]
                header_str = ' '.join(str(h).lower() for h in header if h)

                # --- Separate common price block ---
                if 'common price' in header_str or (
                    not any(is_brand(h) for h in header)
                    and any(is_product_row(clean_cell(r[0])) for r in table[1:] if r)
                ):
                    rows = []
                    for row in table[1:]:
                        cells = [clean_cell(c) for c in row if clean_cell(c)]
                        if len(cells) >= 2:
                            try:
                                rows.append({
                                    'product': cells[0],
                                    'common_price': float(cells[1])
                                })
                            except ValueError:
                                pass
                    if rows:
                        df_summary = pd.DataFrame(rows)
                    continue

                # --- Must have at least one brand column ---
                if not any(is_brand(h) for h in header):
                    continue

                product_col = next(
                    (i for i, h in enumerate(header)
                     if h and 'product' in h.lower()), 0
                )

                # Map paired brand columns (each brand spans 2 consecutive cols)
                brand_col_map = {}
                i = 0
                while i < len(header):
                    cell = header[i]
                    if cell and is_brand(cell):
                        brand_col_map[cell.upper()] = {'min': i, 'max': i + 1}
                        i += 2
                    else:
                        i += 1

                if not brand_col_map:
                    continue

                for row in table[1:]:
                    product = clean_cell(row[product_col]) if product_col < len(row) else None
                    if not is_product_row(product):
                        continue

                    for brand, cols in brand_col_map.items():
                        try:
                            price_min = float(clean_cell(row[cols['min']])) if cols['min'] < len(row) else None
                            price_max = float(clean_cell(row[cols['max']])) if cols['max'] < len(row) else None
                        except (ValueError, TypeError):
                            price_min, price_max = None, None

                        if price_min or price_max:
                            records.append({
                                'product': product.upper(),
                                'brand': brand,
                                'price_min': price_min,
                                'price_max': price_max,
                            })

    df_detail = pd.DataFrame(records) if records else None
    if df_detail is not None:
        df_detail = df_detail.dropna(subset=['price_min', 'price_max'], how='all')
    return df_detail, df_summary

In [13]:
def parse_type_c_single(pdf_path):
    """
    Type C - 2017 late / 2018:
    Single table, no area breakdown, paired brand columns (min, max).
    Overall Range and Common Price ARE columns inside the table.
    """
    records = []
    df_summary = None

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            tables = page.extract_tables({
                "vertical_strategy": "lines",
                "horizontal_strategy": "lines",
                "snap_tolerance": 5,
            })

            for table in tables:
                if not table or len(table) < 2:
                    continue

                header = [clean_cell(c) for c in table[0]]
                header_str = ' '.join(str(h).lower() for h in header if h)

                # Must have overall range or common price as a column
                if 'overall' not in header_str and 'common' not in header_str:
                    continue

                if not any(is_brand(h) for h in header):
                    continue

                # Locate special columns
                overall_min_col = next(
                    (i for i, h in enumerate(header)
                     if h and 'overall' in h.lower()), None
                )
                common_col = next(
                    (i for i, h in enumerate(header)
                     if h and 'common' in h.lower()), None
                )
                product_col = next(
                    (i for i, h in enumerate(header)
                     if h and 'product' in h.lower()), 0
                )

                # Map brand columns, skipping special cols
                brand_col_map = {}
                i = 0
                while i < len(header):
                    cell = header[i]
                    if should_skip_col(cell):
                        i += 1
                        continue
                    if cell and is_brand(cell):
                        brand_col_map[cell.upper()] = {'min': i, 'max': i + 1}
                        i += 2
                    else:
                        i += 1

                overall_summary = []
                for row in table[1:]:
                    product = clean_cell(row[product_col]) if product_col < len(row) else None
                    if not is_product_row(product):
                        continue

                    # Brand prices
                    for brand, cols in brand_col_map.items():
                        try:
                            price_min = float(clean_cell(row[cols['min']])) if cols['min'] < len(row) else None
                            price_max = float(clean_cell(row[cols['max']])) if cols['max'] < len(row) else None
                        except (ValueError, TypeError):
                            price_min, price_max = None, None

                        if price_min or price_max:
                            records.append({
                                'product': product.upper(),
                                'brand': brand,
                                'price_min': price_min,
                                'price_max': price_max,
                            })

                    # Overall range + common price
                    try:
                        o_min  = float(clean_cell(row[overall_min_col])) if overall_min_col is not None else None
                        o_max  = float(clean_cell(row[overall_min_col + 1])) if overall_min_col is not None else None
                        c_price = float(clean_cell(row[common_col])) if common_col is not None else None
                    except (ValueError, TypeError):
                        o_min, o_max, c_price = None, None, None

                    overall_summary.append({
                        'product': product.upper(),
                        'overall_min': o_min,
                        'overall_max': o_max,
                        'common_price': c_price,
                    })

                if overall_summary:
                    df_summary = pd.DataFrame(overall_summary)

    df_detail = pd.DataFrame(records) if records else None
    if df_detail is not None:
        df_detail = df_detail.dropna(subset=['price_min', 'price_max'], how='all')
    return df_detail, df_summary

In [14]:
def parse_type_de(pdf_path, common_price_position='last'):
    """
    Types D and E — area-based formats.
    D (2019-2021): Common Price is the FIRST column after product.
    E (2024-2026): Common Price is the LAST column.
    """
    records = []
    df_summary = None

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            tables = page.extract_tables({
                "vertical_strategy": "lines",
                "horizontal_strategy": "lines",
                "snap_tolerance": 4,
                "join_tolerance": 4,
            })

            for table in tables:
                kind = classify_table(table)

                # --- Summary table ---
                if kind == 'summary':
                    rows = []
                    for row in table[1:]:
                        cells = [clean_cell(c) for c in row if clean_cell(c)]
                        if len(cells) >= 3:
                            try:
                                rows.append({
                                    'product': cells[0],
                                    'overall_min': float(cells[1]),
                                    'overall_max': float(cells[2]),
                                    'common_price': float(cells[3]) if len(cells) > 3 else None
                                })
                            except ValueError:
                                pass
                    if rows:
                        df_summary = pd.DataFrame(rows)
                    continue

                if kind != 'detail':
                    continue

                header = [clean_cell(c) for c in table[0]]

                # Locate fixed columns
                area_col    = next((i for i, h in enumerate(header) if h and 'area'    in h.lower()), None)
                product_col = next((i for i, h in enumerate(header) if h and 'product' in h.lower()), None)

                if area_col is None or product_col is None:
                    continue

                # Map brand columns — skip non-brand cols
                brand_col_map = {}
                i = 0
                while i < len(header):
                    cell = header[i]
                    if should_skip_col(cell) or i in (area_col, product_col):
                        i += 1
                        continue
                    if cell and is_brand(cell):
                        brand_col_map[cell.upper()] = {'min': i, 'max': i + 1}
                        i += 2
                    else:
                        i += 1

                current_area = None
                for row in table[1:]:
                    area_val    = clean_cell(row[area_col])    if area_col    < len(row) else None
                    product_val = clean_cell(row[product_col]) if product_col < len(row) else None

                    if area_val and not is_product_row(area_val):
                        current_area = area_val

                    if not is_product_row(product_val):
                        continue

                    for brand, cols in brand_col_map.items():
                        try:
                            price_min = float(clean_cell(row[cols['min']])) if cols['min'] < len(row) else None
                            price_max = float(clean_cell(row[cols['max']])) if cols['max'] < len(row) else None
                        except (ValueError, TypeError):
                            price_min, price_max = None, None

                        if price_min or price_max:
                            records.append({
                                'area':      current_area,
                                'product':   product_val.upper(),
                                'brand':     brand,
                                'price_min': price_min,
                                'price_max': price_max,
                            })

    df_detail = pd.DataFrame(records) if records else None
    if df_detail is not None:
        df_detail['area'] = df_detail['area'].ffill()
        df_detail = df_detail.dropna(subset=['price_min', 'price_max'], how='all')
    return df_detail, df_summary

In [15]:
def detect_format(pdf_path):
    """
    Returns 'A', 'B', 'C', 'D', or 'E'.

    A - 2017 early   : split tables, XX-XX range strings, no areas
    B - 2017 mid     : single table, no areas, paired cols, NO overall range / common price cols
    C - 2017 late/18 : single table, no areas, paired cols, WITH overall range + common price cols
    D - 2019-2021    : area-based, common price FIRST column
    E - 2024-2026    : area-based, common price LAST column
    """
    with pdfplumber.open(pdf_path) as pdf:
        page  = pdf.pages[0]
        text  = (page.extract_text() or '').lower()
        tables = page.extract_tables()

        # --- Type E: modern DOE format ---
        if 'price monitoring of liquid fuels' in text or 'bagong pilipinas' in text:
            return 'E'

        # --- Type A: range strings like "42.20-47.85" ---
        for t in tables:
            for row in t[1:]:
                for cell in row:
                    if cell and re.search(r'\d+\.\d+-\d+\.\d+', str(cell)):
                        return 'A'

        # --- Area-based check ---
        has_areas = any(
            keyword in str(c).lower()
            for t in tables
            for r in t[:3]
            for c in r if c
            for keyword in ['area', 'cities']
        )

        if has_areas:
            # D vs E: check position of 'common price' vs first brand in header
            for t in tables:
                if not t:
                    continue
                header_flat = [clean_cell(c) or '' for c in t[0]]
                common_pos = next(
                    (i for i, h in enumerate(header_flat) if 'common' in h.lower()), 999
                )
                brand_pos = next(
                    (i for i, h in enumerate(header_flat) if is_brand(h)), 999
                )
                if common_pos < brand_pos:
                    return 'D'
            return 'D'  # fallback

        # --- Single table: B vs C ---
        for t in tables:
            if not t:
                continue
            header_str = ' '.join(str(c).lower() for c in t[0] if c)
            if 'overall' in header_str or 'common' in header_str:
                return 'C'

        return 'B'

### Execution Pipeline

In [16]:
def parse_bulletin(pdf_path):
    pdf_path = str(pdf_path)
    fmt = detect_format(pdf_path)
    effectivity_date = extract_effectivity_date(pdf_path)
    year = effectivity_date.year if effectivity_date else None

    print(f"  → Format: {fmt} | Effectivity: {effectivity_date}")

    if fmt == 'A':
        df_detail, df_summary = parse_type_a(pdf_path)
    elif fmt == 'B':
        df_detail, df_summary = parse_type_b(pdf_path)
    elif fmt == 'C':
        df_detail, df_summary = parse_type_c_single(pdf_path)
    elif fmt == 'D':
        df_detail, df_summary = parse_type_de(pdf_path, common_price_position='first')
    else:
        df_detail, df_summary = parse_type_de(pdf_path, common_price_position='last')

    for df in [df_detail, df_summary]:
        if df is not None:
            df['effectivity_date'] = effectivity_date
            df['year']             = year
            df['source']           = Path(pdf_path).name
            if df is df_detail:
                df['format'] = fmt

    return {
        'format':           fmt,
        'effectivity_date': effectivity_date,
        'detail':           df_detail,
        'summary':          df_summary,
    }

In [17]:
def run_all(pdf_list):
    results = {}
    for path in pdf_list:
        name = Path(path).name
        print(f"Processing: {name}")
        try:
            results[name] = parse_bulletin(path)
            detail = results[name]['detail']
            count  = len(detail) if detail is not None else 0
            print(f"  → {count} rows extracted\n")
        except Exception as e:
            print(f"  → ERROR: {e}\n")
            results[name] = {
                'format': None,
                'effectivity_date': None,
                'detail': None,
                'summary': None
            }
    return results

In [22]:
def build_pdf_list(base_folder_path):
    pdf_list = []
    base_folder = Path(base_folder_path)

    if not base_folder.exists():
        print(f"Base folder not found: {base_folder}")
        return pdf_list

    for subfolder in base_folder.iterdir():
        if subfolder.is_dir():
            pdfs = sorted([
                str(file) 
                for file in subfolder.iterdir()
                if file.is_file() and file.suffix.lower() == '.pdf'
            ])
            
            print(f"{len(pdfs)} files found in {subfolder.name}")
            pdf_list.extend(pdfs)

    print(f"\nTotal PDFs to process: {len(pdf_list)}")
    return pdf_list

In [25]:
target_directory = Path("../data/raw/doe_pdfs_v2")

In [26]:
pdf_list = build_pdf_list(target_directory)
results = run_all(pdf_list)


111 files found in Format 1
235 files found in Format 2
79 files found in Format 3
14 files found in Format 4
11 files found in Format 5

Total PDFs to process: 450
Processing: ncr-price-monitoring-01062026.pdf
  → Format: E | Effectivity: 2026-01-06
  → 0 rows extracted

Processing: ncr-price-monitoring-01132026.pdf
  → Format: E | Effectivity: 2026-01-13
  → 0 rows extracted

Processing: ncr-price-monitoring-01202026.pdf
  → Format: E | Effectivity: 2026-01-20
  → 0 rows extracted

Processing: ncr-price-monitoring-01272026.pdf
  → Format: E | Effectivity: 2026-01-27
  → 0 rows extracted

Processing: ncr-price-monitoring-02032026.pdf
  → Format: E | Effectivity: 2026-02-03
  → 0 rows extracted

Processing: ncr-price-monitoring-02102026-1.pdf
  → Format: E | Effectivity: 2026-02-10
  → 0 rows extracted

Processing: ncr-price-monitoring-02172026.pdf
  → Format: E | Effectivity: 2026-02-17
  → 0 rows extracted

Processing: ncr-price-monitoring-02242026.pdf
  → Format: E | Effectivity: 20

KeyboardInterrupt: 

### Timeline Inspection

In [30]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style for the notebook
sns.set_theme(style="whitegrid")

# Load the data
df_pump = pd.read_parquet('../data/semi-cleaned/processed_doe_data.parquet') 
df_brent = pd.read_parquet('../data/semi-cleaned/brent_weekly.parquet')

# 1. Fix DOE Pump Dates (handle typos like 'Decembeer')
df_pump['Effectivity Date'] = pd.to_datetime(df_pump['Effectivity Date'], format='mixed', errors='coerce')
df_pump = df_pump.dropna(subset=['Effectivity Date'])

# 2. Fix Brent Dates (handle 'week_monday' being the Index)
# We use .index to access the dates shown in your screenshot
df_brent.index = pd.to_datetime(df_brent.index, errors='coerce')
df_brent = df_brent.dropna(how='all') # Clean up any unparseable index rows

# 3. Print Final Date Ranges
print(f"DOE Pump Prices Date Range: {df_pump['Effectivity Date'].min().date()} to {df_pump['Effectivity Date'].max().date()}")
print(f"Brent Crude Date Range: {df_brent.index.min().date()} to {df_brent.index.max().date()}")

DOE Pump Prices Date Range: 0001-02-18 to 2026-04-28
Brent Crude Date Range: 2017-01-09 to 2026-05-04


In [24]:
# 1. Load the data using your new file name
doe_path = '../data/semi-cleaned/processed_doe_data.parquet' # Adjust path if it's in a subfolder
df_pump = pd.read_parquet(doe_path)

# 2. Print the exact column names
print("Your columns are:")
print(df_pump.columns.tolist())

# 3. Look at the first few rows
display(df_pump.head(3))


Your columns are:
['Monitoring Dates', 'Effectivity Date', 'City', 'Product', 'Brand', 'Price Low (P/L)', 'Price High (P/L)', 'Notes']


,Monitoring Dates,Effectivity Date,City,Product,Brand,Price Low (P/L),Price High (P/L),Notes
0,"April 27-28, 2021","Apr 27, 2021",Caloocan City,RON 91,Petron,52.40,52.40,
1,"April 27-28, 2021","Apr 27, 2021",Caloocan City,RON 91,Shell,56.85,56.85,
2,"April 27-28, 2021","Apr 27, 2021",Caloocan City,RON 91,Caltex,47.40,47.40,


In [26]:
df_pump = pd.read_parquet(brent_path)

# 2. Print the exact column names
print("Your columns are:")
print(df_pump.columns.tolist())

# 3. Look at the first few rows
display(df_pump.head(3))

Your columns are:
['brent_close', 'brent_mean']


,brent_close,brent_mean
week_monday,,
2017-01-09,55.90,55.1275
2017-01-16,54.37,54.0160
2017-01-23,55.04,54.1940
